In [1]:
import scipy.io
from scipy import signal
import numpy as np
from scipy.io import savemat
import os

## Uniformize Pipeline Output

Python implementation of MATLAB pipeline uniformization.

**Input**: MATLAB table (N x 7) with columns [G1,G2,G3,G6,G7,G8,G9]  
**Output**: Uniformized table (N x 7) where each cell is (targetLength x 24)

In [2]:
def uniformize_pipeline_output(input_file_path, target_length, output_file_path=None):
    """
    Uniformizes gesture data from MATLAB pipeline output to consistent length.
    
    Parameters:
    -----------
    input_file_path : str
        Path to .mat file containing pipeline output table (N x 7)
        
    target_length : int
        Target number of samples for uniformization
        
    output_file_path : str, optional
        Path to save uniformized data. If None, only returns the data.
    
    Returns:
    --------
    uniformized_table : numpy.ndarray
        Uniformized gesture table (N x 7) where each cell is (targetLength x 24)
    """
    
    # Load the MATLAB file
    print(f"Loading data from: {input_file_path}")
    mat_data = scipy.io.loadmat(input_file_path)
    table_key = 'combinedCell'
    
    gesture_table = mat_data[table_key]
    print(table_key)
    num_rows, num_cols = gesture_table.shape
    
    print(f"Table shape: {num_rows} rows x {num_cols} columns")
    print(f"Target length: {target_length} samples")
    
    # Display statistics
    # display_length_statistics(gesture_table)
    
    # Create output table
    uniformized_table = np.empty_like(gesture_table, dtype=object)
    
    # Process each cell
    total_gestures = 0
    for row in range(num_rows):
        for col in range(num_cols):
            gesture_data = gesture_table[row, col]
            
            # Skip empty cells
            if gesture_data is None or not isinstance(gesture_data, np.ndarray) or gesture_data.size == 0:
                uniformized_table[row, col] = None
                continue
            
            # Resample
            uniformized_table[row, col] = resample_gesture(gesture_data, target_length)
            total_gestures += 1
    
    print(f"\nProcessed {total_gestures} gesture repetitions")
    print(f"All gestures now: ({target_length} x 24)")
    
    # Save if output path provided
    if output_file_path is not None:
        os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
        savemat(output_file_path, {table_key: uniformized_table})
        print(f"Saved to: {output_file_path}")
    return uniformized_table


def resample_gesture(gesture_data, target_length):
    """
    Resamples a single gesture repetition using scipy.signal.resample.
    
    Parameters:
    -----------
    gesture_data : numpy.ndarray
        Original EMG data (M x 24)
        
    target_length : int
        Target number of samples
    
    Returns:
    --------
    resampled_data : numpy.ndarray
        Resampled EMG data (targetLength x 24)
    """
    
    orig_length, num_channels = gesture_data.shape
    
    # If already at target length, return as is
    if orig_length == target_length:
        return gesture_data
    
    # Resample each channel
    resampled_data = np.zeros((target_length, num_channels))
    for ch in range(num_channels):
        resampled_data[:, ch] = signal.resample(gesture_data[:, ch], target_length)
    
    return resampled_data


def display_length_statistics(gesture_table):
    """
    Displays statistics about gesture lengths.
    """
    
    num_rows, num_cols = gesture_table.shape
    gesture_names = ["G1", "G2", "G3", "G6", "G7", "G8", "G9"]
    
    print("\n=== Gesture Length Statistics ===")
    
    for col in range(num_cols):
        lengths = []
        for row in range(num_rows):
            gesture_data = gesture_table[row, col]
            if gesture_data is not None and isinstance(gesture_data, np.ndarray) and gesture_data.size > 0:
                lengths.append(gesture_data.shape[0])
        
        if lengths:
            gesture_name = gesture_names[col] if col < len(gesture_names) else f"G{col}"
            print(f"{gesture_name}: Min={min(lengths)}, Max={max(lengths)}, "
                  f"Mean={np.mean(lengths):.1f}, Median={np.median(lengths):.0f} (n={len(lengths)})")
    
    print("=================================\n")

## Usage Example

In [42]:
# Example usage
input_path = "/Users/chrisdollo/Documents/Research/Data/processed/emg_gestures_combined.mat"
output_path = "/Users/chrisdollo/Documents/Research/Data/final/output.mat"

uniformized_data = uniformize_pipeline_output(
    input_file_path=input_path,
    target_length=1500,
    output_file_path=output_path
)

Loading data from: /Users/chrisdollo/Documents/Research/Data/processed/emg_gestures_combined.mat
combinedCell
Table shape: 40 rows x 7 columns
Target length: 1500 samples

=== Gesture Length Statistics ===
G1: Min=5124, Max=15373, Mean=10248.2, Median=10248 (n=40)
G2: Min=5124, Max=15373, Mean=10248.2, Median=10248 (n=40)
G3: Min=5124, Max=15373, Mean=10248.1, Median=10248 (n=40)
G6: Min=5124, Max=15373, Mean=10248.3, Median=10248 (n=40)
G7: Min=5124, Max=15373, Mean=10248.1, Median=10248 (n=40)
G8: Min=5124, Max=15373, Mean=10248.2, Median=10248 (n=40)
G9: Min=5124, Max=15373, Mean=10248.1, Median=10248 (n=40)


Processed 280 gesture repetitions
All gestures now: (1500 x 24)
Saved to: /Users/chrisdollo/Documents/Research/Data/final/output.mat


## Batch Processing

In [3]:
def batch_uniformize(input_dir, output_dir, target_length):
    """
    Batch process multiple .mat files.
    """
    
    os.makedirs(output_dir, exist_ok=True)
    mat_files = [f for f in os.listdir(input_dir) if f.endswith('.mat')]
    
    print(f"Found {len(mat_files)} files\n")
    
    for i, filename in enumerate(mat_files, 1):
        print(f"\n[{i}/{len(mat_files)}] Processing: {filename}")
        
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, f"uniformized_{filename}")
        
        try:
            uniformize_pipeline_output(input_path, target_length, output_path)
            print(f"✓ Success")
        except Exception as e:
            print(f"✗ Failed: {str(e)}")
    
    print(f"\nComplete! Output in: {output_dir}")

In [17]:
batch_uniformize(
    input_dir="E:/Chris/EMG/Data/non_uniform_data_mat",
    output_dir="E:/Chris/EMG/Data/uniform_mat_data_per_subject",
    target_length=1500
)

Found 44 files


[1/44] Processing: emg_gestures_03_combined_non_uniform.mat
Loading data from: E:/Chris/EMG/Data/non_uniform_data_mat\emg_gestures_03_combined_non_uniform.mat
combinedCell
Table shape: 40 rows x 7 columns
Target length: 1500 samples

Processed 280 gesture repetitions
All gestures now: (1500 x 24)
Saved to: E:/Chris/EMG/Data/uniform_mat_data_per_subject\uniformized_emg_gestures_03_combined_non_uniform.mat
✓ Success

[2/44] Processing: emg_gestures_04_combined_non_uniform.mat
Loading data from: E:/Chris/EMG/Data/non_uniform_data_mat\emg_gestures_04_combined_non_uniform.mat
combinedCell
Table shape: 40 rows x 7 columns
Target length: 1500 samples

Processed 280 gesture repetitions
All gestures now: (1500 x 24)
Saved to: E:/Chris/EMG/Data/uniform_mat_data_per_subject\uniformized_emg_gestures_04_combined_non_uniform.mat
✓ Success

[3/44] Processing: emg_gestures_05_combined_non_uniform.mat
Loading data from: E:/Chris/EMG/Data/non_uniform_data_mat\emg_gestures_05_combined_no

## Combine Multiple Uniformized Files

Combine multiple uniformized .mat files into a single file.

In [5]:
def combine_uniformized_files(input_dir, output_file_path, table_key='combinedCell'):
    """
    Combines multiple uniformized .mat files into a single .mat file.
    
    Parameters:
    -----------
    input_dir : str
        Directory containing uniformized .mat files to combine
        
    output_file_path : str
        Path to save the combined .mat file
        
    table_key : str, default='combinedCell'
        Key name for the table in the .mat files
    
    Returns:
    --------
    combined_table : numpy.ndarray
        Combined gesture table with all files stacked vertically
    """
    
    # Find all .mat files
    mat_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.mat')])
    
    if not mat_files:
        raise ValueError(f"No .mat files found in {input_dir}")
    
    print(f"Found {len(mat_files)} files to combine:")
    for f in mat_files:
        print(f"  - {f}")
    
    combined_table = None
    total_rows = 0
    
    # Load and stack each file
    for i, filename in enumerate(mat_files, 1):
        file_path = os.path.join(input_dir, filename)
        print(f"\n[{i}/{len(mat_files)}] Loading: {filename}")
        
        try:
            mat_data = scipy.io.loadmat(file_path)
            
            # Try to find the table key
            if table_key not in mat_data:
                # Try to find first non-metadata key
                found_key = None
                for key in mat_data.keys():
                    if not key.startswith('__'):
                        found_key = key
                        break
                if found_key:
                    print(f"  Using key '{found_key}' instead of '{table_key}'")
                    table_key = found_key
                else:
                    raise ValueError(f"No data table found in {filename}")
            
            table = mat_data[table_key]
            num_rows, num_cols = table.shape
            print(f"  Shape: {num_rows} rows x {num_cols} columns")
            
            # Initialize combined_table with first file
            if combined_table is None:
                combined_table = table
            else:
                # Stack vertically (concatenate rows)
                combined_table = np.vstack([combined_table, table])
            
            total_rows += num_rows
            
        except Exception as e:
            print(f"  ✗ Error loading {filename}: {str(e)}")
            continue
    
    if combined_table is None:
        raise ValueError("No data was successfully loaded")
    
    # Count non-empty cells
    non_empty = 0
    for r in range(combined_table.shape[0]):
        for c in range(combined_table.shape[1]):
            cell = combined_table[r, c]
            if cell is not None and isinstance(cell, np.ndarray) and cell.size > 0:
                non_empty += 1
    
    print(f"\n{'='*60}")
    print(f"Combined {len(mat_files)} files")
    print(f"Final shape: {combined_table.shape[0]} rows x {combined_table.shape[1]} columns")
    print(f"Total gesture repetitions: {non_empty}")
    print(f"{'='*60}")
    
    # Save combined file
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    savemat(output_file_path, {table_key: combined_table})
    print(f"\n✓ Saved combined file to: {output_file_path}")
    
    return combined_table

In [6]:
# combine_uniformized_files("E:/Chris/EMG/Data/uniform_mat_data_per_subject", 
#                           "E:/Chris/EMG/Data/put_emg_model_data.mat", 
#                           table_key='combinedCell')

combine_uniformized_files("E:/Chris/EMG/Data/data_for_5_subject", 
                          "E:/Chris/EMG/Data/data_for_5_subject.mat", 
                          table_key='combinedCell')

Found 5 files to combine:
  - uniformized_emg_gestures_03_combined_non_uniform.mat
  - uniformized_emg_gestures_04_combined_non_uniform.mat
  - uniformized_emg_gestures_05_combined_non_uniform.mat
  - uniformized_emg_gestures_06_combined_non_uniform.mat
  - uniformized_emg_gestures_07_combined_non_uniform.mat

[1/5] Loading: uniformized_emg_gestures_03_combined_non_uniform.mat
  Shape: 40 rows x 7 columns

[2/5] Loading: uniformized_emg_gestures_04_combined_non_uniform.mat
  Shape: 40 rows x 7 columns

[3/5] Loading: uniformized_emg_gestures_05_combined_non_uniform.mat
  Shape: 40 rows x 7 columns

[4/5] Loading: uniformized_emg_gestures_06_combined_non_uniform.mat
  Shape: 40 rows x 7 columns

[5/5] Loading: uniformized_emg_gestures_07_combined_non_uniform.mat
  Shape: 40 rows x 7 columns

Combined 5 files
Final shape: 200 rows x 7 columns
Total gesture repetitions: 1400

✓ Saved combined file to: E:/Chris/EMG/Data/data_for_5_subject.mat


array([[array([[ -3.06924109,   2.92673837,   2.58212775, ...,  -5.40426804,
                -16.78746542, -15.64147761],
               [ -5.5476226 ,  -5.52762233,  -4.64312372, ...,  -1.50054776,
                 -0.09934318,   1.63340828],
               [ -4.41112528,  -2.40579604,  -2.39403359, ...,  -3.38869043,
                 -4.58175874,  -4.10400199],
               ...,
               [-11.86670732,   5.75073971,   8.50238313, ...,  -0.22202146,
                 -4.05531998, -10.53262108],
               [-10.08310582,   5.6754085 ,   5.85516614, ...,   1.22203475,
                 -2.82165078, -11.68098075],
               [-21.907869  ,  -8.49695267,  -8.16514996, ..., -15.60423677,
                -32.36265456, -33.57101974]])                               ,
        array([[  3.16388793,  -3.25131872,  -5.30603432, ...,   5.52202244,
                  4.78485705,   7.84870815],
               [ -7.70356489, -20.03921568, -11.17091915, ...,  -2.01822488,
                